# 03 — Model Comparison
Compare a few classifiers before committing to the one used in `ai/model/train.py`.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('../ai/data/processed/cleaned_data.csv')

In [ ]:
numeric = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']
X = df[numeric + categorical]
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
pre = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
])

models = {
    'logreg': LogisticRegression(max_iter=1000),
    'random_forest': RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42),
    'gboost': GradientBoostingClassifier(random_state=42),
}

for name, model in models.items():
    pipe = Pipeline([('pre', pre), ('model', model)])
    pipe.fit(X_train, y_train)
    probs = pipe.predict_proba(X_test)[:, 1]
    print(name, 'ROC AUC:', round(roc_auc_score(y_test, probs), 4))